In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models
from tqdm import tqdm  # 진행률 표시용

# 1. 환경 설정
if torch.backends.mps.is_available():
    device = torch.device("mps")
    print("M4 Pro GPU (MPS) 활성화 완료!")
else:
    device = torch.device("cpu")

# 2. 데이터 전처리 (Augmentation 포함)
# 표지판은 방향이 중요하므로 HorizontalFlip은 제외합니다.
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ColorJitter(brightness=0.3, contrast=0.3), # 어두운 이미지 대비
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# 3. 데이터셋 로드
full_train_dataset = torchvision.datasets.GTSRB(root='./data', split='train', download=True, transform=transform)
test_dataset = torchvision.datasets.GTSRB(root='./data', split='test', download=True, transform=transform)

# 학습 데이터 중 10%를 검증(Validation) 데이터로 분할
train_size = int(0.9 * len(full_train_dataset))
val_size = len(full_train_dataset) - train_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

batch_size = 128
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=4)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=4)

# 4. 모델 정의 (EfficientNet-B0 사용)
model = models.efficientnet_b0(weights='IMAGENET1K_V1')
num_ftrs = model.classifier[1].in_features
model.classifier[1] = nn.Linear(num_ftrs, 43) # GTSRB는 클래스가 43개입니다.
model = model.to(device)

# 5. 손실함수 및 최적화 도구
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.002)
# 정확도 정체 시 lr을 낮춰 95% 고지를 넘게 해주는 스케줄러
scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=2, factor=0.1)

# 6. 학습 루프
num_epochs = 10

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{num_epochs}"):
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    
    train_acc = 100 * correct / total
    
    # Validation 체크
    model.eval()
    val_loss = 0.0
    val_correct = 0
    val_total = 0
    with torch.no_grad():
        for images, labels in val_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            _, predicted = outputs.max(1)
            val_total += labels.size(0)
            val_correct += predicted.eq(labels).sum().item()
    
    val_acc = 100 * val_correct / val_total
    print(f"Loss: {running_loss/len(train_loader):.4f} | Acc: {train_acc:.2f}% | Val Acc: {val_acc:.2f}%")
    
    scheduler.step(val_loss)

# 7. 최종 테스트
print("\n--- Final Test ---")
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = outputs.max(1)
        test_total += labels.size(0)
        test_correct += predicted.eq(labels).sum().item()

print(f"Test Accuracy: {100 * test_correct / test_total:.2f}%")

M4 Pro GPU (MPS) 활성화 완료!


Epoch 1/10: 100%|█████████████████████████████| 188/188 [02:41<00:00,  1.17it/s]


Loss: 0.2650 | Acc: 93.05% | Val Acc: 98.24%


Epoch 2/10: 100%|█████████████████████████████| 188/188 [02:32<00:00,  1.23it/s]


Loss: 0.0395 | Acc: 98.89% | Val Acc: 98.05%


...

Loss: 0.0070 | Acc: 99.82% | Val Acc: 99.92%


Epoch 9/10: 100%|█████████████████████████████| 188/188 [02:33<00:00,  1.23it/s]


Loss: 0.0090 | Acc: 99.76% | Val Acc: 99.40%


Epoch 10/10: 100%|████████████████████████████| 188/188 [02:32<00:00,  1.23it/s]


Loss: 0.0231 | Acc: 99.39% | Val Acc: 99.81%

--- Final Test ---
Test Accuracy: 98.24%
